# Frame JSON to SFS Conversion Example

This notebook shows how to convert lidar and camera data from a per-frame JSON into Scale's SFS format.

You can adjust the steps to fit your own data, which hopefully won't be too different.

In [1]:
from dataclasses import asdict
import glob
import os
import pprint
import re
import sys
from typing import List, Union
from smart_open import smart_open
import json

import numpy as np
import pandas as pd

#  !{sys.executable} -m pip install scale_sensor_fusion_io scale_json_binary

import scale_sensor_fusion_io as sfio
from scale_sensor_fusion_io.model_converters.sfs import to_scene_spec_sfs
from scale_sensor_fusion_io.utils.video_helpers import generate_video
from scale_sensor_fusion_io.validation import parse_and_validate_scene
from scale_json_binary.json_binary_codecs import read_file

In [2]:
USEC_IN_SEC = int(1e6)
DATA_PATH = os.path.expanduser("~/frame_json/")

In [3]:
frames = []

# frames are named 0.json, 1.json, 2.json, ...
for fp in sorted(
    glob.glob(os.path.join(DATA_PATH, "frames", "*.json")),
    key=lambda x: int(os.path.splitext(os.path.basename(x))[0]),
):
    with open(fp, "r") as f:
        frames.append(json.load(f))

assert frames, "No frames loaded"
f"loaded {len(frames)} frames"

'loaded 20 frames'

In [4]:
frames[0].keys()

dict_keys(['images', 'points', 'device_position', 'device_heading'])

In [5]:
frames[0]['images'][0].keys()

dict_keys(['position', 'heading', 'image_url', 'camera_model', 'fx', 'fy', 'cx', 'cy', 'skew', 'k1', 'k2', 'p1', 'p2', 'k3', 'k4', 'k5', 'k6', 'scale_factor', 'imageWidth', 'imageHeight', 'imageUrl'])

In [6]:
frames[0]['points'][0].keys()

dict_keys(['x', 'y', 'z', 'i', 'd', 't'])

## Timestamps

Timestamps must satisfy the following conditions:

1. Use microseconds
2. First timestamp is 0
3. Each lidar frame timestamp is less than or equal to every lidar point timestamp in the frame
4. Timestamps for each frame should be monotonically increasing (make sure your frames are in order)

We satisfy point 2 by subtracting the minimum timestamp from all timestamps in our data.

In [7]:
# this data doesn't have any timestamps, so we'll generate timestamps for each frame
CAMERA_FPS = 10
FRAME_TIME_DELTA = USEC_IN_SEC // CAMERA_FPS

time_offset = 1682454481 * USEC_IN_SEC  # this would normally be the minimum timestamp
timestamps = [i * FRAME_TIME_DELTA for i in range(len(frames))]

## LIDAR

In [8]:
# check that intensities are 0-255
min_intensity: Union[int, float] = min(point['i'] for frame in frames for point in frame["points"])
max_intensity: Union[int, float] = max(point['i'] for frame in frames for point in frame["points"])

print("min intensity", min_intensity)
print("max intensity", max_intensity)

assert min_intensity >= 0, "Unexpected negative intensity"
if max_intensity < 1:
    print("Multiplying intensities by 256")
    for frame in frames:
        for point in frame["points"]:
            point["i"] = int(point["i"] * 256)
elif max_intensity > 255 or max_intensity < 128:
    print("Max intensity is wrong and must be adjusted")
else:
    print("Max intensity seems okay")

min intensity 0
max intensity 254
Max intensity seems okay


In [9]:
def generate_lidar_sensor(
    frames: List[dict], timestamps: List[int], sensor_id: str
) -> sfio.LidarSensor:
    """
    frames: [{
        "device_heading": {
            "w": float,
            "x": float,
            "y": float,
            "z": float,
        },
        "device_position": {
            "x": float,
            "y": float,
            "z": float,
        },
        "points": {
            "d": int,  # sensor id
            "i": int,  # intensity
            "x": float,
            "y": float,
            "z": float,
        },
    }]
    """

    assert len(frames) == len(
        timestamps
    ), f"Lengths do not match: len(frames)={len(frames)} len(timestamps)={len(timestamps)}"

    ### CREATE LIDAR FRAMES ###
    lidar_points = [
        sfio.LidarSensorPoints(
            positions=np.array(
                [[point["x"], point["y"], point["z"]] for point in frame["points"]],
                dtype=np.float32,
            ),
            # LidarSensorPoints.timestamps is optional, but must be in np.uint32.
            # Omitted because we don't have timestamps.
            # timestamps=np.array([point['t'] for point in frame['points']], dtype=np.uint32),

            # intensities should be converted to 0-255
            intensities=np.array(
                [point["i"] for point in frame["points"]], dtype=np.uint8
            ),
        )
        for frame in frames
    ]
    lidar_sensor_frames = [
        sfio.LidarSensorFrame(points=points, timestamp=timestamp)
        for points, timestamp in zip(lidar_points, timestamps)
    ]

    ### LIDAR POSES ###

    # PosePath expects an array of poses with shape (N, 7), where N is the number of poses
    poses = sfio.PosePath(
        data=np.fromiter(
            (
                (
                    frame["device_position"]["x"],
                    frame["device_position"]["y"],
                    frame["device_position"]["z"],
                    frame["device_heading"]["x"],
                    frame["device_heading"]["y"],
                    frame["device_heading"]["z"],
                    frame["device_heading"]["w"],
                )
                for frame in frames
            ),
            dtype=np.dtype((float, 7)),
            count=len(frames),
        ),
        index=timestamps,  # index is required
    )
    assert len(poses) == len(
        frames
    ), f"Lengths do not match: len(poses)={len(poses)} len(frames)={len(frames)}"

    # this dataset has points on a world coordinate system
    return sfio.LidarSensor(
        id=sensor_id, poses=poses, frames=lidar_sensor_frames, coordinates="world"
    )

## Camera

**NOTE:** It is important to use the `scale_sensor_fusion_io.utils.generate_video` utility to create a video from images as it uses a specific encoding. Videos are also better for compression.

In [10]:
def s3_object_url_to_uri(url: str) -> str:
    """Hacky way to convert an AWS S3 Object URL to a URI."""

    m = re.match(
        r"https:\/\/(?P<bucket>[a-z0-9\-_]+)\.s3\.[a-z0-9\-_\.]+\.amazonaws\.com\/(?P<key>[a-z0-9\-_\/]+)$",
        url,
    )
    assert m, "URL did not match expected form"
    return f"s3://{m.group('bucket')}/{m.group('key')}"

In [11]:
CAMERA_WIDTH = 1920
CAMERA_HEIGHT = 1080
CAMERA_FPS = 10


def generate_camera_sensor(
    images: List[dict], timestamps: List[int], sensor_id: str, working_dir: str
) -> sfio.CameraSensor:
    """
    images: [{
        "image_url": str,

        # intrinsics
        "fx": float,
        "fy": float,
        "cx": float,
        "cy": float,
        "imageWidth": int,
        "imageHeight": int,
        "skew": float,
        "scale_factor": float,

        # pose
        "heading": {
            "w": float,
            "x": float,
            "y": float,
            "z": float,
        },
        "position": {
            "x": float,
            "y": float,
            "z": float,
        },

        # distortion (Brown Conrady in this case)
        "camera_model": "brown_conrady",
        "k1": float,
        "k2": float,
        "k3": float,
        "k4": float,
        "k5": float,
        "k6": float,
        "p1": float,
        "p2": float,
    }]
    """

    assert images, f"No images for camera {sensor_id}"
    assert len(images) == len(
        timestamps
    ), f"Lengths do not match: len(images)={len(images)} len(timestamps)={len(timestamps)}"

    ### CAMERA INTRINSICS ###'
    images[0]["model"] = images[0]["camera_model"]
    intrinsics = sfio.CameraIntrinsics(
        fx=images[0]["fx"],
        fy=images[0]["fy"],
        cx=images[0]["cx"],
        cy=images[0]["cy"],
        width=images[0]["imageWidth"],
        height=images[0]["imageHeight"],
        distortion=sfio.CameraDistortion.from_dict(images[0]),
        skew=images[0]["skew"],
        scale_factor=images[0]["scale_factor"],
    )

    ### CAMERA POSES ###

    # PosePath expects an array of poses with shape (N, 7), where N is the number of poses
    poses = sfio.PosePath(
        data=np.fromiter(
            (
                (
                    image["position"]["x"],
                    image["position"]["y"],
                    image["position"]["z"],
                    image["heading"]["x"],
                    image["heading"]["y"],
                    image["heading"]["z"],
                    image["heading"]["w"],
                )
                for image in images
            ),
            dtype=np.dtype((float, 7)),
            count=len(images),
        ),
        index=timestamps,  # index is required
    )
    assert len(poses) == len(
        timestamps
    ), f"Lengths do not match: len(poses)={len(poses)} len(timestamps)={len(timestamps)}"

    ### GENERATE VIDEO ###
    os.makedirs(working_dir, exist_ok=True)
    num_digits = len(str(len(images)))
    for i, image in enumerate(images):
        with smart_open(s3_object_url_to_uri(image["image_url"]), "rb") as o, smart_open(
            os.path.join(working_dir, "image_{i:0{l}d}.jpg".format(i=i, l=num_digits)),
            "wb",
        ) as d:
            d.write(o.read())

    # use this utility function to encode the video correctly
    generate_video(
        image_files=sorted(glob.glob(os.path.join(working_dir, f"image_*.jpg"))),
        target_file=os.path.join(working_dir, "video.mp4"),
        fps=CAMERA_FPS,
    )
    video = sfio.CameraSensorVideo(
        timestamps=timestamps,
        content=np.fromfile(os.path.join(working_dir, "video.mp4"), dtype=np.uint8),
        fps=CAMERA_FPS,
    )

    return sfio.CameraSensor(
        id=sensor_id,
        intrinsics=intrinsics,
        video=video,
        poses=poses,
    )

## Create SFS Scene

In [12]:
lidar_sensor = generate_lidar_sensor(frames, timestamps, "lidar_0")  # type: ignore
camera_sensors = [
    generate_camera_sensor(
        [frame["images"][camera_index] for frame in frames],  # type: ignore
        timestamps,
        f"camera_{camera_index}",
        os.path.join(DATA_PATH, f"camera_{camera_index}"),
    )
    for camera_index in range(len(frames[0]["images"]))
]
sensors = camera_sensors + [lidar_sensor]

# must use microseconds
scene = sfio.Scene(
    sensors=sensors, time_offset=time_offset, time_unit="microseconds"  # type: ignore
)
# convert the Scene object to an sfs object
sfs_scene = to_scene_spec_sfs(scene)

100%|██████████| 20/20 [00:00<00:00, 37.00it/s]


In [13]:
encoder = sfio.JSONBinaryEncoder()
encoder.write_file(os.path.join(DATA_PATH, "scene.sfs"), sfs_scene)  # type: ignore

## Test

The following code will validate an SFS file. You can also use LidarLite to view the scene online by dragging and dropping the file: https://dashboard.scale.com/lidarlite/.

In [14]:
pp = pprint.PrettyPrinter(depth=6)


def test0(sfs_scene_path):
    print("Test 0")
    raw_data = read_file(sfs_scene_path)

    result = parse_and_validate_scene(raw_data)

    if not result.success:
        pp.pprint(asdict(result))
    else:
        print("Scene parsed successfully")

In [15]:
test0(os.path.join(DATA_PATH, "scene.sfs"))

Test 0
Scene parsed successfully
